# 🛠️ Day 2: The Mechanics of Agentic AI
**Focus:** Tools, Binding, and the Manual Execution Loop.

---

### 🎯 Agenda
1. **Built-in Tools:** Leveraging the community ecosystem.
2. **Custom Tools & Pydantic:** Creating robust, type-safe tools.
3. **Binding:** How the LLM "sees" functions.
4. **The Message Stack:** Understanding `HumanMessage`, `AIMessage`, and `ToolMessage`.
5. **Manual Execution:** Building a ReAct loop from scratch (No `AgentExecutor`).

In [1]:
# 1. SETUP
# 1. Setup Environment
from dotenv import load_dotenv
load_dotenv()

True

!pip install pip-system-certs

In [2]:
from langchain_ollama import ChatOllama
# Initialize the "Brain" (LLM)
llm=ChatOllama(model="gpt-oss:120b-cloud")

In [3]:
! pip install -U ddgs

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   -------------------------------------- - 3.9/4.1 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 9.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   --------------------- ------------------ 2.1/3.9 MB 9.6 MB/s eta 0:00:01
   --------------------- ------------------ 2.1/3.9 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 6.6 MB/s  0:00:00

   ---------------------------------------- 0/4 [primp]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ----------------------------- 1/4 [lxml]
   ---------- ------------------

In [4]:
! pip install pip-system-certs

In [3]:

from langchain_community.tools import DuckDuckGoSearchRun

# Create the real search tool
search_tool = DuckDuckGoSearchRun()

# Test it directly (No LLM involved yet)
result = search_tool.invoke("Google gravity easter egg launch date?")
print("✅ Internet Search SUCCESS")
print(result)

✅ Internet Search SUCCESS
A Pac-Man related interactive Google Doodle from 2010 will be shown to users searching for "Google Pacman" or "play Pacman". The American technology company Google has added Easter eggs into many of its products and services, such as Google Search, YouTube, and Android since the 2000s. [1][2] Some easter eggs are created by employees in their 20% time. [3] Google avoids adding Easter eggs to ... Comparative Analysis: Google Gravity vs. Other Tricks Google Gravity belongs to a specific category of "Interruption Tricks" where the UI is fundamentally altered. It is helpful to compare it to other surviving easter eggs to understand its longevity. Learn what Google Gravity is, its history by mr doob, fun facts, and how to use Google Gravity in 2025. A complete guide to Google's classic Easter egg. Most people don't expect the Google homepage to suddenly collapse under the weight of gravity. But that's exactly what happens when you experience Google Gravity —a cleve

In [4]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers. Use this for math operations."""
    return a * b

@tool
def get_employee_email(name: str) -> str:
    """Get the email address of an employee by their full name."""
    # Mock Database
    db = {"Sharad Rajore": "sharad.r@zensar.com", "Alice Smith": "alice@zensar.com"}
    return db.get(name, "Email not found")

print(f"Tool Name: {multiply.name}")
print(f"Tool Description: {multiply.description}")
print(f"Tool Args Schema: {multiply.args}")

Tool Name: multiply
Tool Description: Multiply two integers. Use this for math operations.
Tool Args Schema: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [6]:
# TAVILY SEARCH TOOL - Beginner Friendly Version
# Step 1: Install the Tavily package
#pip install tavily-python

# Step 2: Import libraries
import os
from langchain_community.tools.tavily_search import TavilySearchResults

# Step 3: Get your API key
# Go to https://tavily.com and sign up for a FREE API key
if not os.environ.get("TAVILY_API_KEY"):
    api_key = input("Paste your Tavily API Key here: ")
    os.environ["TAVILY_API_KEY"] = api_key

# Step 4: Create the search tool
tavily_tool = TavilySearchResults(max_results=3)

# Step 5: Test it with a simple search
print("Testing Tavily Search...")
result = tavily_tool.invoke("who won yesterdays west bengal election?")
print("\n✅ Search Results:")
print(result)

Testing Tavily Search...

✅ Search Results:
[{'title': 'West Bengal election results updates: Result on final seat awaited ...', 'url': 'https://www.thehindu.com/elections/west-bengal-assembly/west-bengal-election-results-live-updates-may-4-2026/article70934929.ece', 'content': 'Mr. Chetri won with a margin of over 21,000 votes.\n\nThe former captain of the Indian Hockey team, this election was Mr. Chetri’s political debut.  \n   May 04, 2026 15:48  Watch | Mamata Banerjee heads to counting centre West Bengal Chief Minister Mamata Banerjee is heading to the Sakhawat Memorial counting hall in Kolkata.\n\nThis comes as the BJP has so far won four seats and is leading in 194 others. Meanwhile the TMC has won one seat, and is leading in 88 others.\n\n    \n    \n   May 04, 2026 15:43  BJP wins Bhatar, Asansol Dakshin The Bharatiya Janata Party continued to register initial wins, as vote counting progressed.\n\nAs of 3.40 p.m., the saffron party has won Monteswar, Bhatar, and Asansol Dakshi

## Part 2: Custom Tools (The `@tool` Decorator)
When we need custom logic, we use the `@tool` decorator. 
**CRITICAL:** The docstring is the prompt.

In [8]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers. Use this for math operations."""
    return a * b

@tool
def get_employee_email(name: str) -> str:
    """Get the email address of an employee by their full name."""
    # Mock Database
    db = {"Sharad Rajore": "sharad.rajore@zensar.com", "Alice Smith": "alice@zensar.com"}
    return db.get(name, "Email not found")

print(f"Tool Name: {multiply.name}")
print(f"Tool Description: {multiply.description}")
print(f"Tool Args Schema: {multiply.args}")

Tool Name: multiply
Tool Description: Multiply two integers. Use this for math operations.
Tool Args Schema: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Part 3: Advanced Tools (StructuredTool & Pydantic)
What if we need complex inputs? Like a tool that takes a JSON object?
We use **Pydantic** to enforce data types.

In [9]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

# 1. Define the Schema (The "Contract")
class TicketInput(BaseModel):
    issue_type: str = Field(description="The type of issue: 'Hardware' or 'Software'")
    priority: int = Field(description="Priority level (1-5)")
    description: str = Field(description="Detailed description of the problem")

# 2. Define the Function
def create_it_ticket(issue_type: str, priority: int, description: str) -> str:
    return f"TICKET CREATED: [{issue_type}] Priority {priority} - {description}"

# 3. Create the Tool
ticket_tool = StructuredTool.from_function(
    func=create_it_ticket,
    name="create_ticket",
    description="Use this to log IT support tickets.",
    args_schema=TicketInput
)

# Test it manually to see Pydantic in action
print(ticket_tool.invoke({"issue_type": "Software", "priority": "1", "description": "Laptop crashed"}))

TICKET CREATED: [Software] Priority 1 - Laptop crashed


## Part 4: Binding (The Handshake)
The LLM doesn't know these functions exist yet. We must **BIND** them.
This converts Python functions -> OpenAI JSON Schema.

In [10]:
tools = [multiply, get_employee_email, ticket_tool]

# Create a new LLM instance that KNOWS about the tools
llm_with_tools = llm.bind_tools(tools)

# Let's inspect what happens when we call it
query = "My laptop screen is broken. It's urgent!"
response = llm_with_tools.invoke(query)

#query = "what is 2*2"
#response = llm_with_tools.invoke(query)

print("--- RAW LLM RESPONSE ---")
print(f"Content: {response.content}")
print(f"Tool Calls: {response.tool_calls}")

--- RAW LLM RESPONSE ---
Content: 
Tool Calls: [{'name': 'create_ticket', 'args': {'description': 'User reports that laptop screen is broken and requires urgent replacement or repair.', 'issue_type': 'Hardware', 'priority': 5}, 'id': '89dcdfbe-3b71-43a2-9c9d-c5a37d7334ab', 'type': 'tool_call'}]


## Part 5: The Manual Execution Loop
This is the most important part of Day 2. We will NOT use `AgentExecutor`. We will build the loop manually to understand **Message History**.

**The Cycle:**
1. **HumanMessage:** User input.
2. **AIMessage:** LLM decides to call a tool.
3. **ToolMessage:** We run the tool and pass the result back.
4. **AIMessage:** LLM interprets the result and answers.

In [11]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1. User asks a question
messages = [HumanMessage(content="Who is the pm of India? And multiply 55 * 10.")]

# 2. First LLM Call
ai_msg = llm_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg) # IMPORTANT: Add AI's thought to history

print(f"AI Thought: {ai_msg.tool_calls}")



content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-05-04T17:35:12.639651793Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2393529806, 'load_duration': None, 'prompt_eval_count': 238, 'prompt_eval_duration': None, 'eval_count': 117, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--019df40e-e096-7ca3-81b7-cae400843f99-0' tool_calls=[{'name': 'multiply', 'args': {'a': 55, 'b': 10}, 'id': 'c1d3151c-5219-4e40-b609-17421dd7da2f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 238, 'output_tokens': 117, 'total_tokens': 355}
AI Thought: [{'name': 'multiply', 'args': {'a': 55, 'b': 10}, 'id': 'c1d3151c-5219-4e40-b609-17421dd7da2f', 'type': 'tool_call'}]


In [12]:
# 3. Execute Tools Manually
for tool_call in ai_msg.tool_calls:
    selected_tool = {
        "multiply": multiply, 
        "get_employee_email": get_employee_email
    }[tool_call["name"]]
    
    # Run the tool
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # Create the ToolMessage
    tool_msg = ToolMessage(tool_output, tool_call_id=tool_call["id"])
    messages.append(tool_msg)
    print(f"Executed {tool_call['name']} -> Result: {tool_output}")



Executed multiply -> Result: 550


In [13]:
# 4. Second LLM Call (With full history)
final_response = llm_with_tools.invoke(messages)
print(f"\nFinal Answer: {final_response.content}")


Final Answer: The current Prime Minister of India is **Narendra Modi**.

The product of 55 × 10 is **550**.
